# Trip Extraction — Day-by-Day DuckDB

Processes one day at a time to keep memory footprint small and constant.
Skips dates that already have parquet output (resumable).

## Trip logic
- Poll interval: ~450 seconds
- **Cell handover** (cell_id changes between consecutive polls) = vehicle moving
- **Stationary** = same cell for >= `STATIONARY_POLLS` consecutive polls
- **Trip** = continuous run of handovers between two stationary periods

```
polls:  [C1] [C2] [C3] [C3] [C3] [C4] [C5] [C5] [C5]
        move move move same same move move same same
        |------ TRIP 1 ------|  park  |--- TRIP 2 ---|
```

In [1]:
## Config

import duckdb
import os
from pathlib import Path

HANDOVER_GLOB = "/home/jovyan/data/stage/handover_events/**/*.parquet"
STAGE_DIR = Path("/home/jovyan/data/stage/trips")
TMP_DIR   = "/tmp/duckdb_tmp"

# How many consecutive same-cell polls = stationary
STATIONARY_POLLS = 3      # 3 x 450s = ~22 min

# Minimum cell handovers for a trip to be kept
MIN_HANDOVERS = 4

# Minimum distinct cells visited — filters cell-edge oscillation
MIN_CELLS = 10

MAX_FILES = 30

STAGE_DIR.mkdir(parents=True, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)

print(f"HANDOVER_GLOB={HANDOVER_GLOB}")
print(f"TMP_DIR={TMP_DIR}")
print(f"STATIONARY_POLLS={STATIONARY_POLLS}, MIN_HANDOVERS={MIN_HANDOVERS}, MIN_CELLS={MIN_CELLS}")

HANDOVER_GLOB=/home/jovyan/data/stage/handover_events/**/*.parquet
TMP_DIR=/tmp/duckdb_tmp
STATIONARY_POLLS=3, MIN_HANDOVERS=4, MIN_CELLS=10


In [2]:
## Connect and configure DuckDB

try:
    existing_con = globals().get("con")
    if existing_con is not None:
        existing_con.close()
except (Exception, NameError):
    pass

con = duckdb.connect()
con.execute(f"SET temp_directory='{TMP_DIR}'")
con.execute("SET memory_limit='16GB'")
con.execute("SET preserve_insertion_order=false")
con.execute("SET threads=4")
con.execute("SET enable_progress_bar=false")
con.execute(f"CREATE OR REPLACE VIEW handover_events AS SELECT * FROM read_parquet('{HANDOVER_GLOB}')")

summary = con.execute("""
    SELECT
        COUNT(*)        AS total_rows,
        MIN(event_date) AS first_date,
        MAX(event_date) AS last_date,
        COUNT(DISTINCT event_date) AS n_dates
    FROM handover_events
""").df()
print(summary.to_string(index=False))


 total_rows first_date  last_date  n_dates
 1735661691 2025-01-01 2025-12-31      364


In [3]:
## Get list of dates to process

dates = [
    row[0] for row in
    con.execute("SELECT DISTINCT event_date FROM handover_events ORDER BY event_date").fetchall()
]

print(f"Dates to process: {len(dates)}")
print(f"  First: {dates[0]}")
print(f"  Last:  {dates[-1]}")

Dates to process: 364
  First: 2025-01-01
  Last:  2025-12-31


In [4]:
## Process one day at a time

import time

total_trips = 0
skipped = 0
run_start = time.time()

for i, event_date in enumerate(dates, 1):

    # --- Resumable: skip if parquet already written for this date ---
    out_dir = STAGE_DIR / f"event_date={event_date}"
    if out_dir.exists() and any(out_dir.glob("*.parquet")):
        skipped += 1
        print(f"[{i}/{len(dates)}] SKIP {event_date} (already done)")
        continue

    day_start = time.time()

    # --- Step A: build trips_raw for this date only ---
    con.execute("DROP TABLE IF EXISTS trips_raw")
    con.execute(f"""
    CREATE TABLE trips_raw AS
    WITH
    ranked AS (
        SELECT
            vehicle_id, imsi, event_ts, cell_id, rat, event_date,
            LAG(cell_id) OVER (PARTITION BY vehicle_id ORDER BY event_ts) AS prev_cell,
            CASE
                WHEN cell_id != LAG(cell_id) OVER (PARTITION BY vehicle_id ORDER BY event_ts)
                  OR LAG(cell_id) OVER (PARTITION BY vehicle_id ORDER BY event_ts) IS NULL
                THEN 1 ELSE 0
            END AS cell_changed
        FROM handover_events
        WHERE event_date = '{event_date}'
    ),
    with_run AS (
        SELECT *,
            SUM(cell_changed) OVER (PARTITION BY vehicle_id ORDER BY event_ts) AS cell_run_id
        FROM ranked
    ),
    with_poll_count AS (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY vehicle_id, cell_run_id ORDER BY event_ts) AS same_cell_poll_count
        FROM with_run
    ),
    with_stationary AS (
        SELECT *,
            same_cell_poll_count >= {STATIONARY_POLLS} AS stationary,
            LAG(same_cell_poll_count >= {STATIONARY_POLLS})
                OVER (PARTITION BY vehicle_id ORDER BY event_ts) AS prev_stationary
        FROM with_poll_count
    ),
    with_trip_flags AS (
        SELECT *,
            CASE
                WHEN cell_changed = 1
                 AND (prev_stationary OR prev_stationary IS NULL)
                THEN 1 ELSE 0
            END AS trip_start_flag
        FROM with_stationary
    ),
    with_trip_seq AS (
        SELECT *,
            SUM(trip_start_flag) OVER (PARTITION BY vehicle_id ORDER BY event_ts) AS trip_seq,
            NOT stationary AS in_trip
        FROM with_trip_flags
    )
    SELECT *,
        CASE WHEN NOT stationary
            THEN vehicle_id || '_trip_' || LPAD(CAST(trip_seq AS VARCHAR), 3, '0')
            ELSE NULL
        END AS trip_id
    FROM with_trip_seq
    """)

    # --- Step B: aggregate trip summaries ---
    con.execute("DROP TABLE IF EXISTS trip_agg")
    con.execute("""
    CREATE TABLE trip_agg AS
    SELECT
        vehicle_id, imsi, trip_id, trip_seq, event_date,
        MIN(event_ts)                                      AS trip_start,
        MAX(event_ts)                                      AS trip_end,
        COUNT(*)                                           AS n_events,
        COUNT(DISTINCT cell_id)                            AS n_cells,
        SUM(CASE WHEN cell_changed = 1 THEN 1 ELSE 0 END) AS n_handovers,
        FIRST(cell_id ORDER BY event_ts)                   AS first_cell,
        LAST(cell_id ORDER BY event_ts)                    AS last_cell,
        MODE(rat)                                          AS dominant_rat
    FROM trips_raw
    WHERE in_trip
    GROUP BY vehicle_id, imsi, trip_id, trip_seq, event_date
    """)

    # --- Step C: free trips_raw, build cell sequences ---
    con.execute("DROP TABLE IF EXISTS trips_raw")

    con.execute("DROP TABLE IF EXISTS cell_sequences")
    con.execute(f"""
    CREATE TABLE cell_sequences AS
    SELECT
        h.vehicle_id || '_trip_' || LPAD(CAST(t.trip_seq AS VARCHAR), 3, '0') AS trip_id,
        STRING_AGG(h.cell_id, ' -> ' ORDER BY h.event_ts) AS cell_sequence
    FROM handover_events h
    JOIN trip_agg t
        ON  h.vehicle_id = t.vehicle_id
        AND h.event_ts  >= t.trip_start
        AND h.event_ts  <= t.trip_end
    WHERE h.event_date = '{event_date}'
    GROUP BY 1
    """)

    # --- Step D: join and apply filters, write parquet ---
    out_dir.mkdir(parents=True, exist_ok=True)
    out_file = out_dir / "trips.parquet"

    n_trips = con.execute(f"""
    COPY (
        SELECT
            a.vehicle_id, a.imsi, a.event_date, a.trip_id, a.trip_seq,
            a.trip_start, a.trip_end,
            ROUND(DATEDIFF('second', a.trip_start, a.trip_end) / 60.0, 1) AS duration_minutes,
            a.n_handovers, a.n_cells, a.n_events,
            a.first_cell, a.last_cell, a.dominant_rat,
            s.cell_sequence
        FROM trip_agg a
        LEFT JOIN cell_sequences s ON a.trip_id = s.trip_id
        WHERE a.trip_id    IS NOT NULL
          AND a.n_handovers >= {MIN_HANDOVERS}
          AND a.n_cells     >= {MIN_CELLS}
        ORDER BY vehicle_id, trip_start
    ) TO '{out_file}' (FORMAT PARQUET)
    """).fetchone()[0]

    # --- Cleanup working tables ---
    con.execute("DROP TABLE IF EXISTS trip_agg")
    con.execute("DROP TABLE IF EXISTS cell_sequences")

    total_trips += n_trips
    elapsed = time.time() - day_start
    print(f"[{i}/{len(dates)}] {event_date} → {n_trips:,} trips ({elapsed:.1f}s)")

total_elapsed = time.time() - run_start
print(f"\nDone. {len(dates) - skipped} days processed, {skipped} skipped.")
print(f"Total trips written: {total_trips:,}")
print(f"Total runtime: {total_elapsed/60:.1f} min")

[1/364] 2025-01-01 → 0 trips (8.9s)
[2/364] 2025-01-02 → 0 trips (8.2s)
[3/364] 2025-01-03 → 0 trips (10.2s)
[4/364] 2025-01-04 → 1 trips (15.5s)
[5/364] 2025-01-05 → 0 trips (11.4s)
[6/364] 2025-01-06 → 0 trips (10.3s)
[7/364] 2025-01-07 → 1 trips (11.3s)
[8/364] 2025-01-08 → 0 trips (15.5s)
[9/364] 2025-01-09 → 0 trips (52.8s)
[10/364] 2025-01-10 → 0 trips (9.6s)
[11/364] 2025-01-11 → 0 trips (9.4s)
[12/364] 2025-01-12 → 0 trips (9.0s)
[13/364] 2025-01-13 → 1 trips (8.6s)
[14/364] 2025-01-14 → 0 trips (8.4s)
[15/364] 2025-01-15 → 0 trips (8.9s)
[16/364] 2025-01-17 → 3 trips (8.8s)
[17/364] 2025-01-18 → 3 trips (7.6s)
[18/364] 2025-01-19 → 3 trips (8.1s)
[19/364] 2025-01-20 → 7 trips (8.1s)
[20/364] 2025-01-21 → 4 trips (8.0s)
[21/364] 2025-01-22 → 14 trips (7.7s)
[22/364] 2025-01-23 → 10 trips (7.9s)
[23/364] 2025-01-24 → 5 trips (8.1s)
[24/364] 2025-01-25 → 3 trips (7.9s)
[25/364] 2025-01-26 → 4 trips (8.0s)
[26/364] 2025-01-27 → 14 trips (8.0s)
[27/364] 2025-01-28 → 9 trips (7.9s)


In [5]:
## Register trips table in DuckDB from stage partitions only

TRIPS_STAGE_GLOB = f"{STAGE_DIR}/event_date=*/trips.parquet"
print(f"Loading staged trips from: {TRIPS_STAGE_GLOB}")

con.execute("DROP TABLE IF EXISTS trips")
con.execute(f"""
    CREATE TABLE trips AS
    SELECT *
    FROM read_parquet('{TRIPS_STAGE_GLOB}')
""")

required_cols = {
    "vehicle_id", "imsi", "event_date", "trip_id", "trip_seq",
    "trip_start", "trip_end", "duration_minutes", "n_handovers",
    "n_cells", "n_events", "first_cell", "last_cell", "dominant_rat", "cell_sequence"
}
present_cols = {r[1] for r in con.execute("PRAGMA table_info('trips')").fetchall()}
missing = sorted(required_cols - present_cols)
if missing:
    raise ValueError(f"trips table missing expected columns: {missing}")

con.execute("""
    SELECT
        COUNT(*)                       AS n_trips,
        COUNT(DISTINCT vehicle_id)     AS vehicles,
        SUM(n_handovers)               AS total_handovers,
        ROUND(AVG(duration_minutes),1) AS avg_duration_min,
        MIN(event_date)                AS first_date,
        MAX(event_date)                AS last_date
    FROM trips
""").df()

Loading staged trips from: /home/jovyan/data/stage/trips/event_date=*/trips.parquet


,n_trips,vehicles,total_handovers,avg_duration_min,first_date,last_date
0,2085238,93359,36206445.0,159.7,2025-01-04,2025-12-31


In [6]:
## Sanity checks

print("=== Top 10 most-active vehicles ===")
con.execute("""
    SELECT vehicle_id,
           COUNT(*)         AS n_trips,
           SUM(n_handovers) AS total_handovers
    FROM trips
    GROUP BY vehicle_id
    ORDER BY total_handovers DESC
    LIMIT 10
""").df()

=== Top 10 most-active vehicles ===


,vehicle_id,n_trips,total_handovers
0,0672B8A81103E3A99FA8878E0E81250F,613,12329.0
1,D35EFC990419ADFE51A7427BC53202B6,550,11196.0
2,EFE10DB9261CE8EFD35C9DD16C6C3C81,527,10996.0
3,A1B17FAF6EFAF977AD84EFA0C61304F5,462,10909.0
4,18AE84AD76C67003A516520D6BF49ACC,473,10553.0
5,668C70CE315F299803AB6A00C7658B47,314,10466.0
6,1A052A833F4D5192008C5418EE6891C6,340,10371.0
7,83F8306463E117ED2BEF82E3C82939BD,323,9930.0
8,FE10081C77663C41F20939242E3A0C1B,376,9712.0
9,85AE3998D2019A2AF6218ED3DBF12317,389,9564.0


In [7]:
## Inspect most active vehicle

top_vehicle = con.execute("""
    SELECT vehicle_id FROM trips
    GROUP BY vehicle_id
    ORDER BY SUM(n_handovers) DESC
    LIMIT 1
""").fetchone()[0]

print(f"Vehicle: {top_vehicle}")
con.execute(f"""
    SELECT trip_id, trip_start, trip_end, duration_minutes,
           n_handovers, n_cells, cell_sequence
    FROM trips
    WHERE vehicle_id = '{top_vehicle}'
    ORDER BY trip_start
    LIMIT 20
""").df()

Vehicle: 0672B8A81103E3A99FA8878E0E81250F


,trip_id,trip_start,trip_end,duration_minutes,n_handovers,n_cells,cell_sequence
0,0672B8A81103E3A99FA8878E0E81250F_trip_001,2025-03-04 17:41:34.948,2025-03-04 23:52:35.281,371.0,43.0,38,136804618 -> 136804782 -> 136804782 -> 1368071...
1,0672B8A81103E3A99FA8878E0E81250F_trip_006,2025-03-05 05:23:13.770,2025-03-05 07:15:57.116,112.7,13.0,12,137232044 -> 137232044 -> 136870061 -> 1364256...
2,0672B8A81103E3A99FA8878E0E81250F_trip_011,2025-03-05 12:53:05.325,2025-03-05 23:55:17.410,662.2,79.0,71,137924874 -> 137924887 -> 136710316 -> 1368052...
3,0672B8A81103E3A99FA8878E0E81250F_trip_004,2025-03-06 11:04:30.898,2025-03-06 14:57:49.223,233.3,27.0,27,136560045 -> 136560045 -> 137925038 -> 1372626...
4,0672B8A81103E3A99FA8878E0E81250F_trip_005,2025-03-06 15:13:55.769,2025-03-06 21:41:02.826,387.1,44.0,41,138234376 -> 138163750 -> 138163978 -> 1370260...
5,0672B8A81103E3A99FA8878E0E81250F_trip_005,2025-03-07 04:37:23.227,2025-03-07 06:14:10.817,96.8,11.0,10,136560045 -> 136800430 -> 136707607 -> 1371521...
6,0672B8A81103E3A99FA8878E0E81250F_trip_006,2025-03-07 13:37:37.258,2025-03-07 15:22:25.174,104.8,13.0,13,137924874 -> 137924873 -> 136710317 -> 1376116...
7,0672B8A81103E3A99FA8878E0E81250F_trip_008,2025-03-07 18:35:39.583,2025-03-07 21:49:09.735,193.5,21.0,18,138085641 -> 136817069 -> 136790282 -> 1373063...
8,0672B8A81103E3A99FA8878E0E81250F_trip_001,2025-03-08 00:06:02.251,2025-03-08 01:50:51.577,104.8,10.0,10,137477219 -> 137477219 -> 136877834 -> 1381931...
9,0672B8A81103E3A99FA8878E0E81250F_trip_008,2025-03-10 14:27:07.020,2025-03-10 16:19:50.174,112.7,13.0,12,137478755 -> 136779182 -> 136779180 -> 1367791...


In [8]:
con.execute(f"""
    SELECT trip_id, trip_start, trip_end, duration_minutes,
           n_handovers, n_cells, cell_sequence
    FROM trips
    WHERE vehicle_id = 'A2FCC9AF8E480E683F655628A3E5E38D'
    ORDER BY trip_start
    LIMIT 20
""").df()

,trip_id,trip_start,trip_end,duration_minutes,n_handovers,n_cells,cell_sequence
0,A2FCC9AF8E480E683F655628A3E5E38D_trip_020,2025-05-19 18:32:40.767,2025-05-19 21:54:20.614,201.7,22.0,18,182074121 -> 182074129 -> 182074135 -> 1820741...
1,A2FCC9AF8E480E683F655628A3E5E38D_trip_011,2025-05-20 13:12:36.616,2025-05-20 16:05:23.307,172.8,28.0,13,182061079 -> 182017544 -> 9852418 -> 9852418 -...
2,A2FCC9AF8E480E683F655628A3E5E38D_trip_014,2025-05-21 20:38:35.053,2025-05-21 22:50:49.289,132.2,16.0,14,182033161 -> 182714890 -> 182016264 -> 1820339...
3,A2FCC9AF8E480E683F655628A3E5E38D_trip_008,2025-05-22 20:46:20.868,2025-05-22 22:24:42.778,98.4,15.0,10,182035990 -> 182322198 -> 9852418 -> 182017551...
4,A2FCC9AF8E480E683F655628A3E5E38D_trip_009,2025-05-22 22:40:49.059,2025-05-22 23:53:30.694,72.7,10.0,10,182035978 -> 182322198 -> 182061079 -> 1820610...
5,A2FCC9AF8E480E683F655628A3E5E38D_trip_001,2025-05-23 00:01:33.506,2025-05-23 02:26:27.334,144.9,17.0,17,182712329 -> 181819401 -> 181853464 -> 1827517...
6,A2FCC9AF8E480E683F655628A3E5E38D_trip_003,2025-05-25 10:14:21.650,2025-05-25 15:39:17.954,324.9,42.0,37,185806600 -> 185379849 -> 185163273 -> 1853847...
7,A2FCC9AF8E480E683F655628A3E5E38D_trip_010,2025-05-26 19:12:47.145,2025-05-26 21:29:35.196,136.8,14.0,10,182321936 -> 182321928 -> 182321943 -> 1823219...
8,A2FCC9AF8E480E683F655628A3E5E38D_trip_017,2025-05-28 19:55:42.027,2025-05-28 22:28:28.214,152.8,20.0,11,182035990 -> 182035978 -> 182035977 -> 1818396...
9,A2FCC9AF8E480E683F655628A3E5E38D_trip_014,2025-06-05 19:12:39.192,2025-06-05 22:26:50.963,194.2,15.0,10,182187018 -> 182187032 -> 182187032 -> 1823116...


In [9]:
# Export trips for one vehicle safely (avoid loading all rows into pandas)
from pathlib import Path

target_vehicle = 'A2FCC9AF8E480E683F655628A3E5E38D'
export_dir = Path('/home/jovyan/data/exports/trips')
export_dir.mkdir(parents=True, exist_ok=True)

out_parquet = export_dir / f'{target_vehicle}_all_trips.parquet'
out_excel = export_dir / f'{target_vehicle}_all_trips.xlsx'

total_rows = con.execute(f"""
    SELECT COUNT(*)
    FROM trips
    WHERE vehicle_id = '{target_vehicle}'
""").fetchone()[0]

print(f"Vehicle {target_vehicle} has {total_rows:,} trip rows")

# Full export path that does not materialize all rows in Python memory.
con.execute(f"""
    COPY (
        SELECT
            trip_id, trip_start, trip_end, duration_minutes,
            n_handovers, n_cells, cell_sequence
        FROM trips
        WHERE vehicle_id = '{target_vehicle}'
        ORDER BY trip_start
    ) TO '{out_parquet}' (FORMAT PARQUET)
""")
print(f"Full parquet export written: {out_parquet}")

# Excel export is capped to keep kernel memory stable.
EXCEL_MAX_ROWS = 200_000
if total_rows > EXCEL_MAX_ROWS:
    print(
        f"Capping Excel export to first {EXCEL_MAX_ROWS:,} rows to avoid kernel OOM "
        f"(full data is in parquet)."
    )

excel_df = con.execute(f"""
    SELECT
        trip_id, trip_start, trip_end, duration_minutes,
        n_handovers, n_cells, cell_sequence
    FROM trips
    WHERE vehicle_id = '{target_vehicle}'
    ORDER BY trip_start
    LIMIT {EXCEL_MAX_ROWS}
""").df()

excel_df.to_excel(out_excel, index=False)
print(f"Excel export written: {out_excel} ({len(excel_df):,} rows)")

del excel_df

Vehicle A2FCC9AF8E480E683F655628A3E5E38D has 88 trip rows
Full parquet export written: /home/jovyan/data/exports/trips/A2FCC9AF8E480E683F655628A3E5E38D_all_trips.parquet
Excel export written: /home/jovyan/data/exports/trips/A2FCC9AF8E480E683F655628A3E5E38D_all_trips.xlsx (88 rows)


In [10]:
con.execute("""
    SELECT MIN(event_date), MAX(event_date), COUNT(DISTINCT event_date)
    FROM trips
""").df()

,min(event_date),max(event_date),count(DISTINCT event_date)
0,2025-01-04,2025-12-31,352


In [11]:
## Visualisations (memory-safe + progress logging)

import time
import plotly.express as px
import plotly.graph_objects as go

viz_start = time.time()

def _log(step: str):
    elapsed = time.time() - viz_start
    print(f"[{elapsed:6.1f}s] {step}")

TRIPS_SOURCE = "trips"
_log("Starting visualisations...")

overview = con.execute(f"""
    SELECT
        COUNT(*) AS n_trips,
        COUNT(DISTINCT vehicle_id) AS n_vehicles,
        MIN(event_date) AS first_date,
        MAX(event_date) AS last_date
    FROM {TRIPS_SOURCE}
""").df()

print(
    f"Total trips: {int(overview.loc[0, 'n_trips']):,} | "
    f"Vehicles: {int(overview.loc[0, 'n_vehicles']):,} | "
    f"Range: {overview.loc[0, 'first_date']} to {overview.loc[0, 'last_date']}"
)

# 1) Daily trip volume
_log("Querying daily trip volume...")
daily = con.execute(f"""
    SELECT event_date, COUNT(*) AS n_trips
    FROM {TRIPS_SOURCE}
    GROUP BY event_date
    ORDER BY event_date
""").df()

_log("Rendering daily trip volume chart...")
fig1 = px.line(
    daily, x='event_date', y='n_trips',
    title='Daily Trip Volume — Full Year',
    labels={'event_date': 'Date', 'n_trips': 'Trips'}
)
fig1.update_traces(line_width=1.2)
fig1.update_layout(hovermode='x unified')
fig1.show()

# 2) Hourly departure heatmap
_log("Querying hour x weekday matrix...")
heatmap_data = con.execute(f"""
    SELECT
        EXTRACT(DOW FROM trip_start)::INT AS weekday_n,
        CASE EXTRACT(DOW FROM trip_start)::INT
            WHEN 0 THEN 'Sunday'
            WHEN 1 THEN 'Monday'
            WHEN 2 THEN 'Tuesday'
            WHEN 3 THEN 'Wednesday'
            WHEN 4 THEN 'Thursday'
            WHEN 5 THEN 'Friday'
            ELSE 'Saturday'
        END AS weekday,
        EXTRACT(HOUR FROM trip_start)::INT AS hour,
        COUNT(*) AS n_trips
    FROM {TRIPS_SOURCE}
    GROUP BY 1, 2, 3
    ORDER BY 1, 3
""").df()

pivot = heatmap_data.pivot(index='weekday', columns='hour', values='n_trips').fillna(0)
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
pivot = pivot.reindex([d for d in day_order if d in pivot.index])

_log("Rendering heatmap...")
fig2 = go.Figure(go.Heatmap(
    z=pivot.values,
    x=pivot.columns.tolist(),
    y=pivot.index.tolist(),
    colorscale='Viridis',
    colorbar=dict(title='Trips')
))
fig2.update_layout(
    title='Trip Departures — Hour of Day x Day of Week',
    xaxis_title='Hour',
    yaxis_title='Weekday',
)
fig2.show()

# 3) Trip duration distribution (pre-binned in SQL)
_log("Querying duration histogram bins...")
duration_hist = con.execute(f"""
    WITH clipped AS (
        SELECT duration_minutes
        FROM {TRIPS_SOURCE}
        WHERE duration_minutes BETWEEN 1 AND 240
    )
    SELECT
        FLOOR(duration_minutes / 3)::INT * 3 AS bin_start_min,
        COUNT(*) AS n_trips
    FROM clipped
    GROUP BY 1
    ORDER BY 1
""").df()

_log("Rendering duration distribution...")
fig3 = px.bar(
    duration_hist, x='bin_start_min', y='n_trips',
    title='Trip Duration Distribution (1-240 min, 3-min bins)',
    labels={'bin_start_min': 'Duration bin start (min)', 'n_trips': 'Trips'},
    color_discrete_sequence=['steelblue']
 )
fig3.show()

# 4) Handovers distribution (pre-binned in SQL)
_log("Querying handover histogram bins...")
handover_hist = con.execute(f"""
    WITH clipped AS (
        SELECT n_handovers
        FROM {TRIPS_SOURCE}
        WHERE n_handovers <= 200
    )
    SELECT
        n_handovers AS handover_bin,
        COUNT(*) AS n_trips
    FROM clipped
    GROUP BY 1
    ORDER BY 1
""").df()

_log("Rendering handovers distribution...")
fig4 = px.bar(
    handover_hist, x='handover_bin', y='n_trips',
    title='Handovers per Trip Distribution (<=200)',
    labels={'handover_bin': 'Handovers', 'n_trips': 'Trips'},
    color_discrete_sequence=['coral']
 )
fig4.show()

# 5) RAT mix over time
_log("Querying RAT monthly mix...")
rat_monthly = con.execute(f"""
    SELECT
        STRFTIME(trip_start, '%Y-%m') AS month,
        COALESCE(dominant_rat, 'Unknown') AS dominant_rat,
        COUNT(*) AS n_trips
    FROM {TRIPS_SOURCE}
    GROUP BY 1, 2
    ORDER BY 1, 2
""").df()

_log("Rendering RAT monthly mix...")
fig5 = px.bar(
    rat_monthly, x='month', y='n_trips', color='dominant_rat',
    title='Monthly Trip Volume by Dominant RAT (4G / 5G)',
    labels={'month': 'Month', 'n_trips': 'Trips', 'dominant_rat': 'RAT'},
    barmode='stack'
 )
fig5.update_layout(xaxis_tickangle=-45)
fig5.show()

# 6) Top 20 vehicles by handovers
_log("Querying top 20 vehicles...")
top20 = con.execute(f"""
    SELECT
        vehicle_id,
        COUNT(*) AS n_trips,
        SUM(n_handovers) AS total_handovers
    FROM {TRIPS_SOURCE}
    GROUP BY vehicle_id
    ORDER BY total_handovers DESC
    LIMIT 20
""").df()

_log("Rendering top 20 vehicles chart...")
fig6 = px.bar(
    top20, x='vehicle_id', y='total_handovers', color='n_trips',
    title='Top-20 Vehicles by Total Handovers',
    labels={'vehicle_id': 'Vehicle', 'total_handovers': 'Total Handovers', 'n_trips': 'Trips'},
    color_continuous_scale='Blues'
 )
fig6.update_layout(xaxis_tickangle=-60, xaxis_tickfont_size=9)
fig6.show()

# 7) Duration vs Handovers scatter (small DB-side sample only)
_log("Querying scatter sample (max 5k rows)...")
sample = con.execute(f"""
    SELECT
        duration_minutes,
        n_handovers,
        COALESCE(dominant_rat, 'Unknown') AS dominant_rat
    FROM {TRIPS_SOURCE}
    WHERE duration_minutes BETWEEN 1 AND 240
    USING SAMPLE 5000 ROWS
""").df()

_log(f"Rendering scatter ({len(sample):,} rows)...")
fig7 = px.scatter(
    sample, x='duration_minutes', y='n_handovers',
    color='dominant_rat', opacity=0.4,
    title='Trip Duration vs Handovers (sample)',
    labels={
        'duration_minutes': 'Duration (min)',
        'n_handovers': 'Handovers',
        'dominant_rat': 'RAT'
    }
)
fig7.show()

_log("Visualisations complete.")

[   0.0s] Starting visualisations...
Total trips: 2,085,238 | Vehicles: 93,359 | Range: 2025-01-04 00:00:00 to 2025-12-31 00:00:00
[   0.0s] Querying daily trip volume...
[   0.0s] Rendering daily trip volume chart...


[   0.5s] Querying hour x weekday matrix...
[   0.5s] Rendering heatmap...


[   0.5s] Querying duration histogram bins...
[   0.5s] Rendering duration distribution...


[   0.5s] Querying handover histogram bins...
[   0.6s] Rendering handovers distribution...


[   0.6s] Querying RAT monthly mix...
[   0.6s] Rendering RAT monthly mix...


[   0.6s] Querying top 20 vehicles...
[   0.7s] Rendering top 20 vehicles chart...


[   0.8s] Querying scatter sample (max 5k rows)...
[   0.8s] Rendering scatter (4,382 rows)...


[   0.8s] Visualisations complete.
